In [ ]:
#Here I removed the 'defective_insulator' class. Then I merged the 300 image dataset with 'target dataset' of 1046 images.
#The result saved in 'Merged Dataset' folder.
#Then I did the stratified split.The results are saved in 'Merged_Dataset_Stratified'

In [2]:
!pip install roboflow

from roboflow import Roboflow
import os; rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("tl-target-set-focus").project("eduardos-annotated-photos")
version = project.version(1)
dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to eduardos-annotated-photos-1 in yolov5pytorch:: 100%|████████████████████████████████████| 603/603 [00:00<00:00, 7859.41it/s]


In [3]:
from pathlib import Path

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

for split in ["train", "valid", "test"]:
    label_dir = dataset_root / split / "labels"

    for label_file in label_dir.glob("*.txt"):

        new_lines = []

        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()

                cls = int(parts[0])

                if cls == 1:
                    continue

                # remap classes
                if cls == 2:
                    parts[0] = "1"

                elif cls == 3:
                    parts[0] = "2"

                new_lines.append(" ".join(parts))

        with open(label_file, "w") as f:
            f.write("\n".join(new_lines))

In [4]:
from pathlib import Path

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

empty_count = 0

for split in ["train", "valid", "test"]:
    label_dir = dataset_root / split / "labels"

    for label_file in label_dir.glob("*.txt"):
        if label_file.stat().st_size == 0:
            empty_count += 1

print("Empty label files:", empty_count)

Empty label files: 3


In [5]:
from pathlib import Path

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

for split in ["train", "valid", "test"]:
    label_dir = dataset_root / split / "labels"
    image_dir = dataset_root / split / "images"

    for label_file in label_dir.glob("*.txt"):

        if label_file.stat().st_size == 0:

            # find matching image
            for ext in [".jpg", ".jpeg", ".png"]:
                img = image_dir / f"{label_file.stem}{ext}"

                if img.exists():
                    img.unlink()

            label_file.unlink()

In [10]:
from pathlib import Path

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

classes = set()

for split in ["train", "valid", "test"]:
    label_dir = dataset_root / split / "labels"

    for txt in label_dir.glob("*.txt"):

        with open(txt) as f:
            for line in f:

                line = line.strip()

                if not line:
                    continue

                try:
                    cls = int(line.split()[0])
                    classes.add(cls)

                except ValueError:
                    print(f"Problem file: {txt}")
                    print(f"Line: {line}")

print("Classes found:", sorted(classes))

Classes found: [0, 1, 2]


In [11]:
from pathlib import Path

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

empty_count = 0

for split in ["train", "valid", "test"]:
    label_dir = dataset_root / split / "labels"

    for txt in label_dir.glob("*.txt"):

        if txt.stat().st_size == 0:
            empty_count += 1

print("Empty label files:", empty_count)

Empty label files: 0


In [12]:
from pathlib import Path

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

class_counts = {0: 0, 1: 0, 2: 0}

for split in ["train", "valid", "test"]:
    for txt in (dataset_root / split / "labels").glob("*.txt"):

        with open(txt) as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                cls = int(line.split()[0])
                class_counts[cls] += 1

print(class_counts)

{0: 151, 1: 819, 2: 992}


In [13]:
from pathlib import Path

total = 0

for split in ["train", "valid", "test"]:
    total += len(list((Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1") / split / "images").glob("*")))

print("Total images:", total)

Total images: 297


In [14]:
#Merge Datasets for 297 images
from pathlib import Path
import shutil

src_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

dst_images = src_root / "all_images"
dst_labels = src_root / "all_labels"

dst_images.mkdir(exist_ok=True)
dst_labels.mkdir(exist_ok=True)

for split in ["train", "valid", "test"]:

    img_dir = src_root / split / "images"
    lbl_dir = src_root / split / "labels"

    for img in img_dir.glob("*"):
        shutil.copy2(img, dst_images / img.name)

    for lbl in lbl_dir.glob("*.txt"):
        shutil.copy2(lbl, dst_labels / lbl.name)

print("Done.")

Done.


In [15]:
from pathlib import Path

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

mapping = {
    0: 2,  # Defective_Damper
    1: 4,  # Normal_Damper
    2: 5   # Normal_Insulators
}

for split in ["train", "valid", "test"]:

    label_dir = dataset_root / split / "labels"

    for label_file in label_dir.glob("*.txt"):

        new_lines = []

        with open(label_file) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                old_cls = int(parts[0])

                parts[0] = str(mapping[old_cls])

                new_lines.append(" ".join(parts))

        with open(label_file, "w") as f:
            f.write("\n".join(new_lines))

In [16]:
from pathlib import Path

classes = set()

for split in ["train", "valid", "test"]:
    for txt in (Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")/split/"labels").glob("*.txt"):
        with open(txt) as f:
            for line in f:
                line = line.strip()
                if line:
                    classes.add(int(line.split()[0]))

print(sorted(classes))

[2, 4, 5]


In [18]:
from pathlib import Path
import shutil

src_root = Path("/mnt/sdb/home/REDACTED_USER/Target_Stratified")

dst_root = Path("/mnt/sdb/home/REDACTED_USER/Target_Stratified_All")

(dst_root / "images").mkdir(parents=True, exist_ok=True)
(dst_root / "labels").mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:

    img_dir = src_root / split / "images"
    lbl_dir = src_root / split / "labels"

    for img in img_dir.glob("*"):
        shutil.copy2(img, dst_root / "images" / img.name)

    for lbl in lbl_dir.glob("*.txt"):
        shutil.copy2(lbl, dst_root / "labels" / lbl.name)

print("Done")



from pathlib import Path

merged_root = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset")

(merged_root / "images").mkdir(parents=True, exist_ok=True)
(merged_root / "labels").mkdir(parents=True, exist_ok=True)

print("Created merged dataset folders.")



from pathlib import Path
import shutil

src = Path("/mnt/sdb/home/REDACTED_USER/Target_Stratified_All")
dst = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset")

for img in (src / "images").glob("*"):
    shutil.copy2(img, dst / "images" / img.name)

for lbl in (src / "labels").glob("*.txt"):
    shutil.copy2(lbl, dst / "labels" / lbl.name)

print("Copied 1046-image dataset.")

Done
Created merged dataset folders.
Copied 1046-image dataset.


In [19]:
from pathlib import Path
import shutil

src = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All")
dst = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset")

for img in (src / "images").glob("*"):

    new_name = f"rf_{img.name}"

    shutil.copy2(
        img,
        dst / "images" / new_name
    )

for lbl in (src / "labels").glob("*.txt"):

    new_name = f"rf_{lbl.name}"

    shutil.copy2(
        lbl,
        dst / "labels" / new_name
    )

print("Copied 300-image dataset.")

Copied 300-image dataset.


In [20]:
from pathlib import Path

root = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset")

print("Images:", len(list((root/"images").glob("*"))))
print("Labels:", len(list((root/"labels").glob("*.txt"))))

Images: 1046
Labels: 1046


In [21]:
from pathlib import Path

for folder in [
    "/mnt/sdb/home/REDACTED_USER/Target_Stratified_All",
    "/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All",
    "/mnt/sdb/home/REDACTED_USER/Merged_Dataset"
]:
    p = Path(folder)

    images = len(list((p / "images").glob("*")))
    labels = len(list((p / "labels").glob("*.txt")))

    print(folder)
    print("Images:", images)
    print("Labels:", labels)
    print("-" * 30)

/mnt/sdb/home/REDACTED_USER/Target_Stratified_All
Images: 1046
Labels: 1046
------------------------------
/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All
Images: 0
Labels: 0
------------------------------
/mnt/sdb/home/REDACTED_USER/Merged_Dataset
Images: 1046
Labels: 1046
------------------------------


In [22]:
from pathlib import Path

root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")

for split in ["train", "valid", "test"]:
    img_dir = root / split / "images"
    lbl_dir = root / split / "labels"

    print(f"\n{split}")
    print("Images:", len(list(img_dir.glob("*"))))
    print("Labels:", len(list(lbl_dir.glob("*.txt"))))


train
Images: 297
Labels: 297

valid
Images: 0
Labels: 0

test
Images: 0
Labels: 0


In [23]:
from pathlib import Path
import shutil

src_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1")
dst_root = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All")

(dst_root / "images").mkdir(parents=True, exist_ok=True)
(dst_root / "labels").mkdir(parents=True, exist_ok=True)

for img in (src_root / "train" / "images").glob("*"):
    shutil.copy2(img, dst_root / "images" / img.name)

for lbl in (src_root / "train" / "labels").glob("*.txt"):
    shutil.copy2(lbl, dst_root / "labels" / lbl.name)

print("Done")

Done


In [24]:
from pathlib import Path

p = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All")

print("Images:", len(list((p/"images").glob("*"))))
print("Labels:", len(list((p/"labels").glob("*.txt"))))

Images: 297
Labels: 297


In [25]:
from pathlib import Path
import shutil

src = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All")
dst = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset")

for img in (src / "images").glob("*"):
    shutil.copy2(img, dst / "images" / f"rf_{img.name}")

for lbl in (src / "labels").glob("*.txt"):
    shutil.copy2(lbl, dst / "labels" / f"rf_{lbl.name}")

print("Done")

Done


In [26]:
from pathlib import Path

root = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset")

print("Images:", len(list((root/"images").glob("*"))))
print("Labels:", len(list((root/"labels").glob("*.txt"))))

Images: 1343
Labels: 1343


In [29]:
from pathlib import Path
from collections import Counter

labels_dir = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset/labels")

counter = Counter()

for txt in labels_dir.glob("*.txt"):
    with open(txt) as f:
        for line in f:
            line = line.strip()
            if line:
                cls = int(line.split()[0])
                counter[cls] += 1

print(counter)

Counter({4: 2279, 5: 1865, 6: 554, 3: 434, 2: 264, 0: 241, 1: 194})


In [32]:
from pathlib import Path
from collections import Counter

labels_dir = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset/labels")

instance_counts = Counter()

for txt in labels_dir.glob("*.txt"):

    with open(txt) as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            cls = int(line.split()[0])
            instance_counts[cls] += 1

class_names = {
    0: "Birdnest",
    1: "Broken_Insulator",
    2: "Defective_Damper",
    3: "Flashover_Insulator",
    4: "Normal_Damper",
    5: "Normal_Insulators",
    6: "Self-Exploded_Insulator"
}

print("Instance counts:")
print("-" * 40)

total = 0

for cls_id in sorted(instance_counts):
    count = instance_counts[cls_id]
    total += count

    print(f"{cls_id}: {class_names[cls_id]:25s} {count}")

print("-" * 40)
print(f"Total instances: {total}")

Instance counts:
----------------------------------------
0: Birdnest                  241
1: Broken_Insulator          194
2: Defective_Damper          264
3: Flashover_Insulator       434
4: Normal_Damper             2279
5: Normal_Insulators         1865
6: Self-Exploded_Insulator   554
----------------------------------------
Total instances: 5831


In [30]:
from pathlib import Path
from collections import Counter

labels_dir = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All/labels")

counter = Counter()

for txt in labels_dir.glob("*.txt"):
    with open(txt) as f:
        for line in f:
            line = line.strip()
            if line:
                cls = int(line.split()[0])
                counter[cls] += 1

class_names = {
    0: "Birdnest",
    1: "Broken_Insulator",
    2: "Defective_Damper",
    3: "Flashover_Insulator",
    4: "Normal_Damper",
    5: "Normal_Insulators",
    6: "Self-Exploded_Insulator"
}

for cls_id in sorted(counter):
    print(f"{cls_id}: {class_names[cls_id]} = {counter[cls_id]}")

2: Defective_Damper = 151
4: Normal_Damper = 819
5: Normal_Insulators = 992


In [31]:
from pathlib import Path
from collections import Counter

labels_dir = Path("/mnt/sdb/home/REDACTED_USER/eduardos-annotated-photos-1_All/labels")

instance_counts = Counter()

for txt in labels_dir.glob("*.txt"):

    with open(txt) as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            cls = int(line.split()[0])
            instance_counts[cls] += 1

class_names = {
    0: "Birdnest",
    1: "Broken_Insulator",
    2: "Defective_Damper",
    3: "Flashover_Insulator",
    4: "Normal_Damper",
    5: "Normal_Insulators",
    6: "Self-Exploded_Insulator"
}

print("Instance counts:")
print("-" * 40)

total = 0

for cls_id in sorted(instance_counts):
    count = instance_counts[cls_id]
    total += count

    print(f"{cls_id}: {class_names[cls_id]:25s} {count}")

print("-" * 40)
print(f"Total instances: {total}")

Instance counts:
----------------------------------------
2: Defective_Damper          151
4: Normal_Damper             819
5: Normal_Insulators         992
----------------------------------------
Total instances: 1962


In [34]:
from pathlib import Path
from collections import Counter

labels_dir = Path("/mnt/sdb/home/REDACTED_USER/Target_Stratified_All/labels")

instance_counts = Counter()

for txt in labels_dir.glob("*.txt"):

    with open(txt) as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            cls = int(line.split()[0])
            instance_counts[cls] += 1

class_names = {
    0: "Birdnest",
    1: "Broken_Insulator",
    2: "Defective_Damper",
    3: "Flashover_Insulator",
    4: "Normal_Damper",
    5: "Normal_Insulators",
    6: "Self-Exploded_Insulator"
}

print("Instance counts in 1046-image dataset")
print("-" * 50)

total = 0

for cls_id in range(7):
    count = instance_counts.get(cls_id, 0)
    total += count
    print(f"{cls_id}: {class_names[cls_id]:25s} {count}")

print("-" * 50)
print(f"Total instances: {total}")

Instance counts in 1046-image dataset
--------------------------------------------------
0: Birdnest                  241
1: Broken_Insulator          194
2: Defective_Damper          113
3: Flashover_Insulator       434
4: Normal_Damper             1460
5: Normal_Insulators         873
6: Self-Exploded_Insulator   554
--------------------------------------------------
Total instances: 3869


In [35]:
from pathlib import Path

labels_dir = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset/labels")

multi_class_images = 0

for txt in labels_dir.glob("*.txt"):

    classes = set()

    with open(txt) as f:
        for line in f:
            line = line.strip()

            if line:
                classes.add(int(line.split()[0]))

    if len(classes) > 1:
        multi_class_images += 1

print("Multi-class images:", multi_class_images)

Multi-class images: 757


In [37]:
!pip install iterative-stratification
from pathlib import Path
import shutil
import yaml
import numpy as np

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# SETTINGS

dataset_root = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset")

output_root = Path("/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified")

split_ratios = (0.70, 0.15, 0.15)

random_seed = 42

num_classes = 7


# CREATE OUTPUT FOLDERS


for split in ["train", "val", "test"]:

    (output_root / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (output_root / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )


# COLLECT IMAGES + MULTI-LABEL TARGETS

all_rows = []
all_targets = []

images_dir = dataset_root / "images"
labels_dir = dataset_root / "labels"

for img_path in images_dir.glob("*"):

    label_path = labels_dir / f"{img_path.stem}.txt"

    if not label_path.exists():
        continue

    class_vector = np.zeros(num_classes, dtype=int)

    with open(label_path) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            cls = int(line.split()[0])

            if 0 <= cls < num_classes:
                class_vector[cls] = 1

    # skip images with no labels
    if class_vector.sum() == 0:
        continue

    all_rows.append(
        (img_path, label_path)
    )

    all_targets.append(class_vector)

all_targets = np.array(all_targets)

print(f"\nTotal labeled images: {len(all_rows)}")


# TRAIN vs TEMP

indices = np.arange(len(all_rows))

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=(1 - split_ratios[0]),
    random_state=random_seed
)

train_idx, temp_idx = next(
    msss.split(indices, all_targets)
)

print("\nAfter first split")
print("Train:", len(train_idx))
print("Temp :", len(temp_idx))


# VAL vs TEST

temp_targets = all_targets[temp_idx]

val_fraction_of_temp = (
    split_ratios[1] /
    (split_ratios[1] + split_ratios[2])
)

msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=(1 - val_fraction_of_temp),
    random_state=random_seed
)

temp_local_idx = np.arange(len(temp_idx))

val_local_idx, test_local_idx = next(
    msss2.split(temp_local_idx, temp_targets)
)

val_idx = temp_idx[val_local_idx]
test_idx = temp_idx[test_local_idx]

print("\nFinal split")
print("Train:", len(train_idx))
print("Val  :", len(val_idx))
print("Test :", len(test_idx))


# COPY FILES

def copy_split(indices, split_name):

    for idx in indices:

        img_path, label_path = all_rows[idx]

        shutil.copy2(
            img_path,
            output_root /
            split_name /
            "images" /
            img_path.name
        )

        shutil.copy2(
            label_path,
            output_root /
            split_name /
            "labels" /
            label_path.name
        )

copy_split(train_idx, "train")
copy_split(val_idx, "val")
copy_split(test_idx, "test")


# CREATE YAML

yaml_dict = {
    "path": str(output_root.resolve()),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 7,
    "names": [
        "Birdnest",
        "Broken_Insulator",
        "Defective_Damper",
        "Flashover_Insulator",
        "Normal_Damper",
        "Normal_Insulators",
        "Self-Exploded_Insulator"
    ]
}

with open(
    output_root / "merged_stratified.yaml",
    "w"
) as f:
    yaml.safe_dump(
        yaml_dict,
        f,
        sort_keys=False
    )

print("\nFinished creating stratified dataset.")
print(f"Dataset location: {output_root}")


Total labeled images: 1343

After first split
Train: 936
Temp : 407

Final split
Train: 936
Val  : 199
Test : 208

Finished creating stratified dataset.
Dataset location: /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified
